# 01 — The position network

Exploration of the aggregate transition network across all `final` matches: PageRank hubs,
Markov reward-risk, communities, and fighter similarity. All logic lives in
`analysis.network_metrics` / `analysis.fighter_similarity`; this notebook just visualises.
Run from the repo root (`uv run jupyter lab`); needs a `.env` with the DB connection.

In [ ]:
from dotenv import load_dotenv; load_dotenv()
import pandas as pd, matplotlib.pyplot as plt, networkx as nx
from db.base import db_session
from analysis.network_metrics import (build_transition_network, node_centralities,
    reward_risk_ranking, detect_communities, route_to_submission, pagerank_ranking)
from analysis.fighter_similarity import athlete_style_vectors, nearest_in

with db_session() as s:
    G = build_transition_network(s)
    ids, mat = athlete_style_vectors(s)
print(G.number_of_nodes(), 'positions,', G.number_of_edges(), 'transition types')

## Hubs of grappling — PageRank

In [ ]:
pd.DataFrame(pagerank_ranking(G, 15), columns=['position', 'pagerank'])

## Reward-risk balance (Lamas et al. 2024)

In [ ]:
rr = reward_risk_ranking(G, min_occ=5, limit=15)
for n, _, _ in rr[:6]:
    print(n, ' → ', ' → '.join(route_to_submission(G, n)[1:]))
pd.DataFrame(rr, columns=['position', 'reward_risk', 'seen'])

## Game families — community detection

In [ ]:
for i, c in enumerate(detect_communities(G, min_occ=3)[:6], 1):
    print(f'Family {i} ({len(c)}):', ', '.join(c[:10]))

## The network — top-30 positions, sized by PageRank

In [ ]:
cent = node_centralities(G)
top = [n for n, _ in pagerank_ranking(G, 30)]
H = G.subgraph(top)
pos = nx.spring_layout(H, seed=1, k=0.6)
sizes = [cent[n]['pagerank'] * 18000 for n in H.nodes]
plt.figure(figsize=(13, 10))
nx.draw_networkx_edges(H, pos, alpha=0.2, arrows=False)
nx.draw_networkx_nodes(H, pos, node_size=sizes, node_color='#4d86ff', alpha=0.8)
nx.draw_networkx_labels(H, pos, font_size=8)
plt.axis('off'); plt.title('Grappling position network (top 30 by PageRank)'); plt.show()

## Fighter DNA — stylistically closest athletes

In [ ]:
for _, name in ids[:8]:
    print(name, '→', nearest_in(ids, mat, name, 3))